# Volatility Dynamics 16 — Calibrating AM-T variance scale and density shape by horizon

**Experimental status: Historical exploratory; not prospective confirmation.** Frozen methodology: [VD 033](../../discussions/Volatility_Dynamics/033_chatgpt_vd16_horizon_calibration_preregistration.md).

This is **not** fitting Student-t to volatility: native five-minute returns still follow the *unchanged* VD15 Student-t transition process. A horizon-specific **forecast-density overlay** rescales predicted variance, adjusts mixture tail shape at constant mean variance, or does both. As a result, the calibrated forecasts are **not guaranteed to be a single horizon-consistent Markov process**.

Five-minute Gaussian/Student-t Markov innovations are unchanged; no future targets are used to produce a historical forecast. All test labels were already examined by earlier modules, so results remain exploratory.

Seven horizons: **5m, 10m, 15m, 30m, 60m, 120m, 240m**. Primary: 60m. Candidates: `AM_T`, `AM_T_SCALE`, `AM_T_SHAPE`, `AM_T_BOTH`, `AM_G` and `M2_student_t`.


## 0. Repository, environment and immutable protocol

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

ROOT=Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run the notebook within deribit-market-data repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as vd14
from data_engineering.modeling import student_t_return_markov as vd15
from data_engineering.modeling import markov_return_horizon_calibration as vd16

DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT in .env to the historic data root'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), f'Cannot find historic five-minute prices: {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'16'
OUT.mkdir(exist_ok=True,parents=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
PARTICLES=int(os.getenv('VD16_PARTICLES','256'))
BOOT=int(os.getenv('VD16_BOOTSTRAPS','1000'))
assert PARTICLES==vd16.PARTICLES and BOOT==vd16.BOOT, 'Binding spec uses 256 particles and 1000 bootstrap draws'
print(vd16.VERSION,'| HISTORICAL_EXPLORATORY | source',HIST)
print(json.dumps(vd16.registration(),indent=2,default=str))
START=time.monotonic()

## 1. Scientific unit tests before loading real market data

In [ ]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
    str(ROOT/'tests'/'test_markov_return_horizon_calibration.py')],
    cwd=str(ROOT),text=True,capture_output=True)
print(p.stdout[-4000:],p.stderr[-1500:])
assert p.returncode==0, 'VD16 tests must pass before market-data evaluation' 

## 2. Causal five-minute data and seven exact future targets

The historical input is exclusively Binance spot close-to-close five-minute prices. Native 5m returns are censored across missing bars, target windows never cross TRAIN/VALID/TEST partitions, and the already-inspected sample ends strictly before 2026-09-19 UTC.

In [ ]:
raw,excluded=vd14.load_returns(HIST,ASSETS)
panel=vd14.panelize(raw)
del raw
features=vd14.make_features(panel)
frame=vd14.make_targets(panel,features,vd16.HORIZONS)
print('Excluded assets:',excluded)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size'),
    valid_return=('return_5m','count')).to_string())
print('Prepared in',round(time.monotonic()-START,1),'seconds')

## 3. Fit calibrated AM-T distributions and score TEST (historically inspected)

The original AM-T five-minute return kernel is trained on TRAIN and selects five-minute degrees of freedom using VALID. Horizon calibration parameters are learned on **early VALID**, with embargo before **late VALID**, which supplies model-family selection scores. Neither calibration stage uses TEST labels. Scoring remains exploratory because the same historical TEST period influenced prior research.

In [ ]:
scores,boot,asset_scores,calibration_grid,params=vd16.run_all(
    frame,particles=PARTICLES,B=BOOT,cal_limit=vd16.CAL_ORIGINS_PER_ASSET)
print('Models per horizon:')
print(scores.groupby('horizon_min').agg(models=('model','nunique'),
    origins=('origins','first'),days=('test_days','first')).to_string())
assert len(scores)==len(vd16.HORIZONS)*len(vd16.MODEL_NAMES)
assert (scores.groupby('horizon_min').origins.nunique()==1).all()
print('Fitted and scored in',round(time.monotonic()-START,1),'seconds')

## 4. Calibration settings, 7-horizon scores and paired bootstrap

In [ ]:
display(calibration_grid.round(6))
display(scores.round(7))
display(boot[(boot.horizon_min==vd16.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))].round(7))
display(asset_scores.loc[asset_scores.horizon_min==vd16.PRIMARY].round(7))
print('WARNING: historical TEST was already inspected in VD13-15; these are descriptive results, not validation of a new edge.')

## 5. Check numerical quadrature stability using VALID only

The 256-particle mixture approximation can make apparent small density gains. Compare 256 versus nested 512 particles at 5m, 60m and 240m using **late VALID only**. Do not use historical TEST results to alter quadrature settings.

In [ ]:
particle_check=vd16.particle_sensitivity_validation(frame,
    params['native_return_kernels'],params['state_transitions'],params['calibrations'],
    horizons=(5,60,240),particles=PARTICLES)
display(particle_check.round(8))

## 6. Historical-only research decision and compact saved outputs

In [ ]:
decision=vd16.decision(scores,boot,params)
assert decision['status']=='HISTORICAL_EXPLORATORY'
assert decision['prospective_confirmation'] is False
print(json.dumps(decision,indent=2))

scores.to_csv(OUT/'16_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'16_bootstrap.csv',index=False,float_format='%.10g')
asset_scores.to_csv(OUT/'16_per_asset.csv',index=False,float_format='%.10g')
calibration_grid.to_csv(OUT/'16_calibration_grid.csv',index=False,float_format='%.10g')
particle_check.to_csv(OUT/'16_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'16_decision.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_16','version':vd16.VERSION,
          'registration':vd16.registration(),'history_directory':str(HIST),
          'sample_start':str(panel.state_time.min()),'sample_end':str(panel.state_time.max()),
          'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
          'split_days':panel.groupby('split').date.nunique().to_dict(),
          'parameters_and_selection':params,
          'decision':decision,
          'scientific_caveat':'Already inspected historical test; no new confirmation or trading claim.'}
(OUT/'16_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved seven compact results files to:',OUT)
print('No origin-level predictions, raw market data or DuckDB modifications.')